# Assignment 4: Pipelines and Hyperparameter Tuning (48 total marks)
### Due: March 20 at 11:59pm

### Name: Vihang Shah

The purpose of this assignment is to practice following the grid-search workflow: 
- Split data into training and test set
- Use the training portion to find the best model using grid search and cross-validation
- Retrain the best model
- Evaluate the retrained model on the test set

In [39]:
import numpy as np
import pandas as pd

## Part 1: Classification (21 marks)

### 1.1: Load data (1 mark)
For this task, we will be using the yellowbrick mushroom dataset. This dataset uses physical characteristics of mushrooms to predict whether or not the mushroom is poisonous.

More information on the dataset can be found here:
https://www.scikit-yb.org/en/latest/api/datasets/mushroom.html

#### Prepare the feature matrix and target vector

Using the yellowbrick `load_mushroom()` function, load the mushroom data set into feature matrix `X` and target vector `y`.

Print the shape of `X` and `y`.

In [40]:
# TO DO: Load the dataset and print the shape of X and y (0.5 marks)
from yellowbrick.datasets import load_mushroom

X, y = load_mushroom()
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (8123, 3)
y shape: (8123,)


In [41]:
# TO DO: Inspect the first few lines of code (0.5 marks)
X.head()

,shape,surface,color
0,convex,smooth,yellow
1,bell,smooth,white
2,convex,scaly,white
3,convex,smooth,gray
4,convex,scaly,yellow


### 1.2: Pre-processing (2 marks)
In this dataset, all the features are categorical, so they need to be encoded. We will use `OneHotEncoder(sparse_output=False)` for this case.

In [42]:
# TO DO: Create OneHotEncoder object (0.5 marks)
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

encoder.fit(X)
X_encoded = encoder.transform(X)
X_encoded.shape

(8123, 20)

The next step is to build a pipeline to combine the encoding with the selected machine learning method. To initialize the pipeline, we will use `LogisticRegression(max_iter=1000)` as a placeholder.

In [43]:
# TO DO: Build the pipeline (1 mark)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline(steps=[
    ("encoder", encoder),
    ("classifier", LogisticRegression(max_iter=1000))
])

pipe

,steps,"[('encoder', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None


The next step is to split the data into training and testing sets. Use `test_size=0.1, stratify=y, random_state=42`.

In [44]:
# TO DO: Split data into training and testing sets (0.5 marks)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    stratify=y,
    random_state=42
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (7310, 3)
Test shape: (813, 3)


### 1.3: Grid Search (4 marks)

For the grid search, we would like to test three different models: `LogisticRegression(max_iter=1000)`, `KNeighborsClassifier()` and `SVC()`. Build your parameter grid based on what you think are reasonable values to test.

In [45]:
# TO DO: Build a parameter grid (3 marks - one for each classifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

param_grid = [
    {
        "classifier": [LogisticRegression(max_iter=1000)],
        "classifier__C": [0.01, 0.1, 1.0, 10.0]
    },
    {
        "classifier": [KNeighborsClassifier()],
        "classifier__n_neighbors": [3, 5, 7, 9],
        "classifier__weights": ["uniform", "distance"]
    },
    {
        "classifier": [SVC()],
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__gamma": ["scale", "auto"]
    }
]

In [46]:
# TO DO: Implement grid search (1 mark)
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"[{'classifier': [LogisticRegre...max_iter=1000)], 'classifier__C': [0.01, 0.1, ...]}, {'classifier': [KNeighborsClassifier()], 'classifier__n_neighbors': [3, 5, ...], 'classifier__weights': ['uniform', 'distance']}, ...]"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,categories,'auto'


### 1.4: Visualize Results (2 marks)

The final step is to print out the results from the grid search. You will need to print out the following items:
- Best parameters
- Best cross-validation train score 
- Best cross-validation test score
- Test set accuracy

In [47]:
# TO DO: Print the results from the grid search (0.5 marks for each correct print statement)

print("Best parameters:", grid.best_params_)
print("Best cross-validation train score:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Best cross-validation test score:", grid.best_score_)
print("Test set accuracy:", grid.score(X_test, y_test))

Best parameters: {'classifier': SVC(), 'classifier__C': 10.0, 'classifier__gamma': 'scale'}
Best cross-validation train score: 0.7178522571819427
Best cross-validation test score: 0.7138166894664842
Test set accuracy: 0.6912669126691267


### Questions (8 marks)

1. How many columns does the encoded dataset have? In general, what is one negative to adding more columns?
1. Which model and what parameters produced the best results?
1. Was this model a good fit? Why or why not?
1. Is there anything else we could do to try to improve model performance? Provide two ideas.

*ANSWER HERE*
1. The encoded dataset contains 20 columns. Adding more columns directly increases the dimensionality of the feature space. Higher dimensionality slows training, increases memory usage, and makes models more prone to overfitting because they can begin to memorize noise rather than learn general patterns. It can also degrade the performance of distance‑based algorithms such as KNN, which become less effective as the number of dimensions grows.
2. The best model found by your grid search was an `SVC` with a regularization parameter `C = 10.0` and `gamma = 'scale'`. This combination produced the highest cross‑validation performance among all tested models, outperforming both Logistic Regression and KNN within the parameter ranges you selected.
3. The model is not a particularly good fit. Although the training and validation scores are close (indicating low overfitting), the overall accuracy is only around 70%, which suggests the model is underfitting and not capturing enough complexity in the data.
4. Model performance could be improved by expanding the hyperparameter search space, such as testing additional values of C, experimenting with different kernels, or exploring a wider range of gamma values. Another way to improve performance would be to try different model families altogether, such as Random Forests or Gradient Boosting classifiers, which often perform strongly on categorical datasets and may capture patterns that SVC does not.

### Process Description (4 marks)
Please describe the process you used to create your code. Cite any websites or generative AI tools used. You can use the following questions as guidance:
1. Where did you source your code?
1. In what order did you complete the steps?
1. If you used generative AI, what prompts did you use? Did you need to modify the code at all? Why or why not?
1. Did you have any challenges? If yes, what were they? If not, what helped you to be successful?

*DESCRIBE YOUR PROCESS HERE - BE SPECIFIC*
1. Most, if not all, of the code is primarily my own. I did, however, derived influences from the Pipelines & Preprocessing lecture, where we learned to use Pipeline, OneHotEncoder, and avoid data leakage by combining preprocessing with the model. The structure of the grid search came directly from the Grid Search & Cross‑Validation lecture, which showed how to build parameter grids and compare multiple classifiers. The coding pattern followed the Breast Cancer dataset pipeline example, which demonstrated swapping classifiers inside a pipeline, and the KNN / SVC / Logistic Regression examples, which guided the hyperparameters we tested. The idea of using the mushroom dataset and one‑hot encoding all features came from the Yellowbrick Mushroom dataset example shown in lecture.
2. In the order listed in the notebook. Like Step 1, then Step 2, then Step 3, etc. Following the notebook order helped ensure that each model was trained and evaluated consistently before moving on to the next experiment.
3. I used GenAI to refine my code where necessary, asking things like <i>`Can you help clean up or improve this code script?`</i>, and finally to fix the grammar in my written answers <i>`Can you fix the grammar in my answers without changing the tone or sentence structure?`</i>. I did need to slightly modify the generated code to better fit the structure of my notebook and match the variables I was already using.
4. I encountered no major challenges while completing this assignment. My success was due to extensive preparation from class examples and lectures, which involved reviewing all lecture examples and practice notebooks provided in the course.

# Part 2: Regression (25 marks)

For this task, we will be using the auto-mpg dataset again. The dataset can be found here: https://archive.ics.uci.edu/ml/datasets/Auto%2BMPG

### 2.1: Load data (2 marks)

#### Prepare the feature matrix and target vector

Using the code below, load the dataset and separate it into feature matrix `X` and target vector `y`. Which column represents the target vector?

Print the shape of `X` and `y`.

In [48]:
# TO DO: Read in the dataset (0.5 marks)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

column_names = [
    'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 
    'acceleration', 'model year', 'origin', 'car name'
]

df = pd.read_csv(url, names=column_names, sep='\s+', na_values='?')

In [49]:
# TO DO: Separate dataset into feature matrix and target vector (0.5 marks)
X = df.drop(columns=['mpg'])
y = df['mpg']

# TO DO: Print shape and type of X and y (0.5 marks)
print("X shape:", X.shape, "type:", type(X))
print("y shape:", y.shape, "type:", type(y))

X shape: (398, 8) type: <class 'pandas.core.frame.DataFrame'>
y shape: (398,) type: <class 'pandas.core.series.Series'>


Do we have any missing values in this case?

In [50]:
# TO DO: Check if there are any missing values (0.5 marks)
missing_before = df.isnull().sum()
print(missing_before)

df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())

missing_after = df.isnull().sum()
print(missing_after)

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model year      0
origin          0
car name        0
dtype: int64
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model year      0
origin          0
car name        0
dtype: int64


### 2.2: Pre-processing (5 marks)
In this dataset, we have a mixture of categorical and numerical data. This means that we will need to use a `ColumnTransformer()`

For this assignment, we will remove the `car_name` column.

In [51]:
# TO DO: Remove car_name column (0.5 marks)
X = X.drop(columns=['car name'])

X.head()

,cylinders,displacement,horsepower,weight,acceleration,model year,origin
0,8,307.0,130.0,3504.0,12.0,70,1
1,8,350.0,165.0,3693.0,11.5,70,1
2,8,318.0,150.0,3436.0,11.0,70,1
3,8,304.0,150.0,3433.0,12.0,70,1
4,8,302.0,140.0,3449.0,10.5,70,1


For this case, we will use:
- `OneHotEncoder(sparse_output=False)` for any categorical columns
- `StandardScaler()` for any numerical columns
- Minimal information imputation for any missing values

In [52]:
# TO DO: Create ColumnTransformer (3 marks)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

categorical_cols = ["origin"]   # treat origin as categorical
numeric_cols = [col for col in X.columns if col not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(sparse_output=False))
])

ct = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

ct

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


The next step is to build a pipeline to combine the ColumnTransformer with the selected machine learning method. To initialize the pipeline, we will use `LinearRegression()` as a placeholder

In [53]:
# TO DO: Build the pipeline (1 mark)
from sklearn.linear_model import LinearRegression

pipe = Pipeline(steps=[
    ("preprocessor", ct),
    ("regressor", LinearRegression())
])

pipe

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


The next step is to split the data into training and testing sets. Use `test_size=0.1, random_state=0`

In [54]:
# TO DO: Split data into training and testing sets (0.5 marks)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=0
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (358, 7)
Test shape: (40, 7)


### 2.3: Grid Search (4 marks)

For the grid search, we would like to test three different models: `LinearRegression()`, `KNeighborsRegressor()` and `RandomForestRegressor(random_state=0)`. Build your parameter grid based on what you think are reasonable values to test

In [55]:
# TO DO: Build a parameter grid (3 marks - one for each model)
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

param_grid = [
    {
        "regressor": [LinearRegression()],
        # LinearRegression has no major hyperparameters to tune
    },
    {
        "regressor": [KNeighborsRegressor()],
        "regressor__n_neighbors": [3, 5, 7, 9],
        "regressor__weights": ["uniform", "distance"]
    },
    {
        "regressor": [RandomForestRegressor(random_state=0)],
        "regressor__n_estimators": [50, 100, 200],
        "regressor__max_depth": [3, 5, 7, None]
    }
]

In [56]:
# TO DO: Implement Grid Search (1 mark)
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

,estimator,Pipeline(step...egression())])
,param_grid,"[{'regressor': [LinearRegression()]}, {'regressor': [KNeighborsRegressor()], 'regressor__n_neighbors': [3, 5, ...], 'regressor__weights': ['uniform', 'distance']}, ...]"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('num', ...), ('cat', ...)]"


### 2.4: Visualize Results (2 marks)

The final step is to print out the results from the grid search. You will need to print out the following items:
- Best parameters
- Best cross-validation train score 
- Best cross-validation test score
- Test set accuracy

In [57]:
# TO DO: Print the results from the grid search (0.5 marks for each correct print statement)
print("Best parameters:", grid.best_params_)
print("Best CV train score:", grid.cv_results_["mean_train_score"][grid.best_index_])
print("Best CV test score:", grid.best_score_)
print("Test set score:", grid.score(X_test, y_test))

Best parameters: {'regressor': RandomForestRegressor(random_state=0), 'regressor__max_depth': 7, 'regressor__n_estimators': 100}
Best CV train score: 0.9695482663005975
Best CV test score: 0.8666135016637252
Test set score: 0.9252701273914445


### Questions (8 marks)

1. Which model and what parameters produced the best results?
1. Was this model a good fit? Why or why not?
1. Is there anything else we could do to try to improve model performance? Provide two ideas (must be different than the two ideas given for the previous part).
1. What happens if you leave the `car_name` column in? Why do you think this happens?

*ANSWER HERE*
1. The best‑performing model in the regression grid search was the RandomForestRegressor with `max_depth = 7` and `n_estimators = 100`. This model achieved a very strong cross‑validation train score of approximately 0.97 and a cross‑validation test score of about 0.87, which was noticeably higher than the performance of both Linear Regression and KNN.
2. The RandomForestRegressor was a strong fit for this dataset because, similar to the classification results in Part 1, the training, validation, and testing scores were all close to one another, indicating that the model did not suffer from significant underfitting or overfitting. However, unlike Part 1, this regression model achieved consistently high performance, with cross‑validation scores in the high 80s and a test score above 92 percent.
3. A simple way to improve performance would be to try different scaling methods for the numerical features, since models like KNN and even Random Forests can behave differently depending on how the data is distributed. Another improvement would be to engineer new features that capture relationships the original variables don’t express directly.
4. If the car_name column is left in the dataset, model performance typically gets worse because the column contains hundreds of unique strings that get one‑hot encoded into a very large number of sparse dummy variables. These columns don’t represent meaningful numerical patterns; instead, they act like unique identifiers for each car. As a result, the model will most likely overfit and generalize poorly.

### Process Description (4 marks)
Please describe the process you used to create your code. Cite any websites or generative AI tools used. You can use the following questions as guidance:
1. Where did you source your code?
1. In what order did you complete the steps?
1. If you used generative AI, what prompts did you use? Did you need to modify the code at all? Why or why not?
1. Did you have any challenges? If yes, what were they? If not, what helped you to be successful?

*DESCRIBE YOUR PROCESS HERE - BE SPECIFIC*
1. Part 2 followed the approach shown in the Regression lecture, especially the Auto‑MPG dataset example, which covered handling missing values, removing car name, and treating origin as categorical. The preprocessing structure using ColumnTransformer with numeric and categorical pipelines came from the Adult dataset example, which demonstrated combining SimpleImputer, StandardScaler, and OneHotEncoder in one transformer. The grid search setup for testing multiple regressors reused the same pattern from the Grid Search & Cross‑Validation lecture, simply applied to regression models instead of classifiers.
2. In the order listed in the notebook. Like Step 1, then Step 2, then Step 3, etc. Following the notebook order helped ensure that each model was trained and evaluated consistently before moving on to the next experiment.
3. I used GenAI to refine my code where necessary, asking things like <i>`Can you help clean up or improve this code script?`</i>, and finally to fix the grammar in my written answers <i>`Can you fix the grammar in my answers without changing the tone or sentence structure?`</i>. I did need to slightly modify the generated code to better fit the structure of my notebook and match the variables I was already using.
4. I encountered no major challenges while completing this assignment. My success was due to extensive preparation from class examples and lectures, which involved reviewing all lecture examples and practice notebooks provided in the course.

## Part 3: Reflection (2 marks)
Include a sentence or two about:
- what you liked or disliked,
- found interesting, confusing, challangeing, motivating
while working on this assignment.


*ADD YOUR THOUGHTS HERE*
- Because I had already reviewed the lecture examples in detail, it felt natural and satisfying to move through each section and see the pieces come together into a clean, reproducible pipeline.
- What stood out most was how much easier the process became thanks to the structure I followed: reviewing class material, adapting the patterns from the examples, and refining small pieces with GenAI when needed.